In [1]:
import hashlib
import hmac
import math
import os
import random
import sqlite3
import tempfile
import threading
from datetime import datetime, timedelta

# ------------------------------------------------------------------ CONFIG
EXPERIMENT_ID = "EXP-ONB-RECOVERY-001"
SECRET = b"demo-secret-change-me"          # that su: luu trong secret manager, khong commit len git
ALGORITHM_VERSION = "hmac-sha256-v1"
BUCKETS = 10_000
TREATMENT_BUCKETS = 5_000                    # 50/50
OUTCOME_WINDOW_DAYS = 14                     # EVAL 02

DDL = """
CREATE TABLE IF NOT EXISTS experiment_assignment (
    assignment_id        INTEGER PRIMARY KEY AUTOINCREMENT,
    experiment_id        TEXT NOT NULL,
    applicant_key        TEXT NOT NULL,
    arm                  TEXT NOT NULL CHECK (arm IN ('TREATMENT','CONTROL')),
    index_application_id TEXT NOT NULL,
    hash_bucket          INTEGER NOT NULL,
    assigned_at          TEXT NOT NULL,
    algorithm_version    TEXT NOT NULL,
    UNIQUE (experiment_id, applicant_key)
);
CREATE TABLE IF NOT EXISTS account_events (
    applicant_key  TEXT NOT NULL,
    application_id TEXT NOT NULL,
    event_type     TEXT NOT NULL,
    event_time     TEXT NOT NULL
);
"""


# ------------------------------------------------------------- LOP 1: HASH
def hash_bucket(applicant_key: str, experiment_id: str = EXPERIMENT_ID, secret: bytes = SECRET) -> int:
    msg = f"{experiment_id}|{applicant_key}".encode()
    digest = hmac.new(secret, msg, hashlib.sha256).digest()
    return int.from_bytes(digest[:8], "big") % BUCKETS


def arm_from_bucket(bucket: int) -> str:
    return "TREATMENT" if bucket < TREATMENT_BUCKETS else "CONTROL"


# ------------------------------------------------------- LOP 2: BANG DANG KY
def connect(path: str) -> sqlite3.Connection:
    conn = sqlite3.connect(path, timeout=30)
    conn.executescript(DDL)
    return conn


def assign(conn, applicant_key, application_id, eligible, now=None):
    """
    Tra ve dict (arm, index_application_id, ...) hoac None neu khong du dieu kien.
    - Chi randomize nguoi du dieu kien (EVAL 01).
    - Da co dong dang ky -> tra lai dong cu, KHONG doi nhom, KHONG doi index application.
    - Ho so thu hai cua cung nguoi khong tao dong moi.
    """
    if not eligible:
        return None
    now = now or datetime.now()
    b = hash_bucket(applicant_key)
    conn.execute(
        """INSERT OR IGNORE INTO experiment_assignment
           (experiment_id, applicant_key, arm, index_application_id, hash_bucket, assigned_at, algorithm_version)
           VALUES (?,?,?,?,?,?,?)""",
        (EXPERIMENT_ID, applicant_key, arm_from_bucket(b), application_id, b,
         now.isoformat(timespec="seconds"), ALGORITHM_VERSION),
    )
    conn.commit()
    row = conn.execute(
        """SELECT arm, index_application_id, hash_bucket, assigned_at
           FROM experiment_assignment WHERE experiment_id=? AND applicant_key=?""",
        (EXPERIMENT_ID, applicant_key),
    ).fetchone()
    return dict(arm=row[0], index_application_id=row[1], hash_bucket=row[2], assigned_at=row[3])


def may_send_experimental_reminder(conn, applicant_key) -> bool:
    """Chot chan o cong dispatch cuoi cung: Control khong bao gio nhan nhac nho thu nghiem."""
    row = conn.execute(
        "SELECT arm FROM experiment_assignment WHERE experiment_id=? AND applicant_key=?",
        (EXPERIMENT_ID, applicant_key),
    ).fetchone()
    return row is not None and row[0] == "TREATMENT"


def converted_within_window(conn, applicant_key):
    """Ket qua chinh: co account_opened trong 14 ngay ke tu assigned_at (EVAL 02)."""
    row = conn.execute(
        "SELECT assigned_at, index_application_id FROM experiment_assignment WHERE experiment_id=? AND applicant_key=?",
        (EXPERIMENT_ID, applicant_key),
    ).fetchone()
    if row is None:
        return None
    start = datetime.fromisoformat(row[0])
    end = start + timedelta(days=OUTCOME_WINDOW_DAYS)
    events = conn.execute(
        "SELECT event_time FROM account_events WHERE applicant_key=? AND event_type='account_opened'",
        (applicant_key,),
    ).fetchall()
    return any(start <= datetime.fromisoformat(e[0]) <= end for e in events)


# ------------------------------------------------------------------ TESTS
def chi2_p_value_1dof(chi2):
    return math.erfc(math.sqrt(chi2 / 2))


def run_tests():
    tmp = tempfile.mkdtemp()
    db = os.path.join(tmp, "demo.db")
    conn = connect(db)
    t0 = datetime(2026, 11, 12, 9, 0, 0)

    print("TEST 1 - Tat dinh: cung applicant_key -> cung bucket/nhom")
    assert len({hash_bucket("APL-000123") for _ in range(1000)}) == 1
    print("  PASS")

    print("TEST 2 - Nhieu ho so cua cung mot nguoi -> cung nhom, giu index application dau tien")
    r1 = assign(conn, "APL-000777", "APP-1", True, t0)
    r2 = assign(conn, "APL-000777", "APP-2", True, t0 + timedelta(days=3))
    r3 = assign(conn, "APL-000777", "APP-3", True, t0 + timedelta(days=9))
    assert r1 == r2 == r3 and r1["index_application_id"] == "APP-1"
    n = conn.execute("SELECT COUNT(*) FROM experiment_assignment WHERE applicant_key='APL-000777'").fetchone()[0]
    assert n == 1
    print(f"  PASS (nhom = {r1['arm']}, index = {r1['index_application_id']}, so dong = {n})")

    print("TEST 3 - Chay lai (idempotent) khong doi ket qua, khong them dong")
    before = conn.execute("SELECT COUNT(*) FROM experiment_assignment").fetchone()[0]
    for _ in range(5):
        assign(conn, "APL-000777", "APP-1", True, t0 + timedelta(days=30))
    after = conn.execute("SELECT COUNT(*) FROM experiment_assignment").fetchone()[0]
    assert before == after
    print("  PASS")

    print("TEST 4 - Doi secret/cau hinh hash sau khi da chia: bang dang ky van giu nhom cu (chong nhay nhom)")
    global SECRET
    saved = SECRET
    old = assign(conn, "APL-000777", "APP-9", True, t0)
    SECRET = b"secret-bi-doi"
    new = assign(conn, "APL-000777", "APP-9", True, t0)
    SECRET = saved
    assert old["arm"] == new["arm"]
    print("  PASS")

    print("TEST 5 - Nguoi khong du dieu kien khong duoc chia nhom")
    assert assign(conn, "APL-EXCLUDED", "APP-X", False, t0) is None
    assert conn.execute("SELECT COUNT(*) FROM experiment_assignment WHERE applicant_key='APL-EXCLUDED'").fetchone()[0] == 0
    print("  PASS")

    print("TEST 6 - Dua luong: 50 luong cung chia nhom cho 1 nguoi -> dung 1 dong, 1 nhom")
    results = []

    def worker(i):
        c = connect(db)
        results.append(assign(c, "APL-RACE", f"APP-{i}", True, t0)["arm"])
        c.close()

    ths = [threading.Thread(target=worker, args=(i,)) for i in range(50)]
    [t.start() for t in ths]
    [t.join() for t in ths]
    rows = conn.execute("SELECT COUNT(*) FROM experiment_assignment WHERE applicant_key='APL-RACE'").fetchone()[0]
    assert rows == 1 and len(set(results)) == 1
    print(f"  PASS (so dong = {rows}, nhom = {set(results)})")

    print("TEST 7 - Can bang 100.000 applicant_key gia lap + kiem tra lech ty le (SRM, chi-square)")
    random.seed(42)
    N = 100_000
    t = sum(1 for i in range(N) if arm_from_bucket(hash_bucket(f"APL-{i:07d}")) == "TREATMENT")
    c = N - t
    exp = N / 2
    chi2 = (t - exp) ** 2 / exp + (c - exp) ** 2 / exp
    p = chi2_p_value_1dof(chi2)
    print(f"  Treatment = {t}, Control = {c}, chi2 = {chi2:.3f}, p = {p:.3f}")
    assert p > 0.001
    print("  PASS (p > 0,001: khong co dau hieu lech ty le)")

    print("TEST 8 - Co mau 2.400 nguoi (quy mo pilot): chenh lech giua hai nhom")
    t = sum(1 for i in range(2400) if arm_from_bucket(hash_bucket(f"APL-{i:07d}")) == "TREATMENT")
    print(f"  Treatment = {t}, Control = {2400 - t}  (chenh lech {abs(2*t-2400)} nguoi; do ngau nhien don gian, khong chia dung 1200/1200)")

    print("TEST 9 - Cong dispatch: Control khong bao gio duoc gui nhac nho thu nghiem")
    ctrl = next(k for k in (f"APL-T{i}" for i in range(200)) if arm_from_bucket(hash_bucket(k)) == "CONTROL")
    trt = next(k for k in (f"APL-T{i}" for i in range(200)) if arm_from_bucket(hash_bucket(k)) == "TREATMENT")
    assign(conn, ctrl, "APP-C", True, t0)
    assign(conn, trt, "APP-T", True, t0)
    assert may_send_experimental_reminder(conn, ctrl) is False
    assert may_send_experimental_reminder(conn, trt) is True
    assert may_send_experimental_reminder(conn, "APL-CHUA-CHIA-NHOM") is False
    print("  PASS (nguoi chua chia nhom cung bi chan)")

    print("TEST 10 - Ket qua 14 ngay: bien cua cua so")
    k = "APL-OUT"
    assign(conn, k, "APP-O", True, t0)
    ev = lambda d: conn.execute("INSERT INTO account_events VALUES (?,?,?,?)", (k, "APP-O", "account_opened", (t0 + d).isoformat()))
    ev(timedelta(days=14)); conn.commit()
    assert converted_within_window(conn, k) is True
    conn.execute("DELETE FROM account_events WHERE applicant_key=?", (k,))
    ev(timedelta(days=14, seconds=1)); conn.commit()
    assert converted_within_window(conn, k) is False
    print("  PASS (dung 14 ngay tinh; qua 1 giay thi khong tinh)")

    print("\n=> TAT CA TEST DAT")


if __name__ == "__main__":
    run_tests()

TEST 1 - Tat dinh: cung applicant_key -> cung bucket/nhom
  PASS
TEST 2 - Nhieu ho so cua cung mot nguoi -> cung nhom, giu index application dau tien
  PASS (nhom = CONTROL, index = APP-1, so dong = 1)
TEST 3 - Chay lai (idempotent) khong doi ket qua, khong them dong
  PASS
TEST 4 - Doi secret/cau hinh hash sau khi da chia: bang dang ky van giu nhom cu (chong nhay nhom)
  PASS
TEST 5 - Nguoi khong du dieu kien khong duoc chia nhom
  PASS
TEST 6 - Dua luong: 50 luong cung chia nhom cho 1 nguoi -> dung 1 dong, 1 nhom
  PASS (so dong = 1, nhom = {'CONTROL'})
TEST 7 - Can bang 100.000 applicant_key gia lap + kiem tra lech ty le (SRM, chi-square)
  Treatment = 49767, Control = 50233, chi2 = 2.172, p = 0.141
  PASS (p > 0,001: khong co dau hieu lech ty le)
TEST 8 - Co mau 2.400 nguoi (quy mo pilot): chenh lech giua hai nhom
  Treatment = 1171, Control = 1229  (chenh lech 58 nguoi; do ngau nhien don gian, khong chia dung 1200/1200)
TEST 9 - Cong dispatch: Control khong bao gio duoc gui nhac n